# T303e — Train v3 (BGE-m3 + MNR + checkpointing + best-on-dev)

Production v1.2 recipe (ADR 013: binary + MNR won over CoSENT + graded), plus four discipline upgrades:

| Lever | v1.2 | v3 | Reason |
|---|---|---|---|
| batch size | 16 | **32** | more in-batch negatives → better MNR convergence |
| epochs | 1 | **2** | with checkpointing overfitting is bounded |
| checkpointing | none | **every 200 steps** | resume on crash; pick best-on-dev |
| evaluation | end-only | **every 200 steps** | catch peak before overfit |

**Baseline to beat (v1.2, joint-pool dev):**
- graded nDCG@10: **0.544** (target ≥0.600 for +10%)
- graded Recall@5: **0.479**
- dev MRR (binary): **0.468**

**Runtime:** ~45-55 min on Colab Free T4 (2 epochs × ~22 min + 5 evaluator runs).

## 1. Install + auth

In [ ]:
!pip install -q sentence-transformers==3.4.1 datasets==3.2.0 huggingface_hub==0.27.0

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Pull training pairs (label==1.0 only, per ADR 013)

In [ ]:
from datasets import load_dataset
from collections import Counter

PAIRS_REPO = 'ManmohanBuildsProducts/auto-parts-search-pairs'
ds = load_dataset(PAIRS_REPO, split='train')
positives = ds.filter(lambda r: r['label'] >= 1.0)
print(f'loaded {len(ds)} pairs; {len(positives)} positives')
print(Counter(positives['pair_type']).most_common())

## 3. Pull KG corpus + dev subset for eval

We mirror `training/evaluate.py:load_corpus` — same corpus the production harness uses.

In [ ]:
import json
import urllib.request
import sqlite3
from collections import defaultdict

# Pull graph.db + benchmark_dev.json from HF raw dataset (reproducibility chain from ADR 012).
# For Colab simplicity, we use a lightweight alternative: pull both from GitHub raw.
# If repo is private, paste the files manually via Colab's file upload panel instead.

# Option A: if repo is public, these URLs work
import os, subprocess
if not os.path.exists('graph.db'):
    # Upload graph.db (~5 MB) and benchmark_dev.json to Colab manually:
    # Files panel (left sidebar) -> upload:
    #   data/knowledge_graph/graph.db
    #   data/training/golden/benchmark_dev.json
    print('UPLOAD graph.db and benchmark_dev.json via the Files panel on the left, then re-run this cell.')
    raise SystemExit()

# Mirror training/evaluate.py:load_corpus
conn = sqlite3.connect('graph.db')
parts = list(conn.execute("SELECT id, name FROM nodes WHERE type='part'"))
aliases = defaultdict(list)
for alias_name, part_id in conn.execute(
    "SELECT n.name, e.dst FROM edges e JOIN nodes n ON n.id = e.src "
    "WHERE e.type='known_as' AND n.type='alias'"
):
    aliases[part_id].append(alias_name)
systems = defaultdict(list)
for part_id, sys_name in conn.execute(
    "SELECT e.src, n.name FROM edges e JOIN nodes n ON n.id = e.dst "
    "WHERE e.type='in_system' AND n.type='system'"
):
    systems[part_id].append(sys_name)
conn.close()

part_ids, doc_texts, rel_strings = [], [], []
for pid, name in parts:
    al = aliases.get(pid, [])
    sy = systems.get(pid, [])
    doc = name
    if al: doc += ' | ' + ', '.join(al)
    if sy: doc += ' | system: ' + ', '.join(sy)
    part_ids.append(pid)
    doc_texts.append(doc)
    rel_strings.append({name.lower(), *(a.lower() for a in al)})

print(f'corpus: {len(part_ids)} parts')

benchmark_dev = json.load(open('benchmark_dev.json'))
print(f'dev benchmark: {len(benchmark_dev)} queries')

## 4. Stratified 40-query dev subset for mid-training eval

In [ ]:
import random
from collections import defaultdict as _dd

PER_TYPE = 7  # 6 types × 7 = 42 queries; close enough to 40
rng = random.Random(42)

by_type = _dd(list)
for q in benchmark_dev:
    by_type[q['query_type']].append(q)

dev_subset = []
for qt, qs in sorted(by_type.items()):
    shuffled = list(qs)
    rng.shuffle(shuffled)
    dev_subset.extend(shuffled[:PER_TYPE])
print(f'dev subset: {len(dev_subset)} queries')
print({qt: sum(1 for q in dev_subset if q['query_type']==qt) for qt in by_type})

## 5. Build InformationRetrievalEvaluator

Same relevance rule as `training/evaluate.py`: a part is relevant if any `expected_parts` string substring-matches the part's name or any alias.

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# queries: {qid: text}
queries_map = {f'q{i}': q['query'] for i, q in enumerate(dev_subset)}
# corpus: {docid: text}
corpus_map = {pid: text for pid, text in zip(part_ids, doc_texts)}
# relevant_docs: {qid: set of docids}
relevant_docs = {}
for i, q in enumerate(dev_subset):
    expected = [ep.lower().strip() for ep in q.get('expected_parts', []) if ep.strip()]
    rels = set()
    for pid, rel_str_set in zip(part_ids, rel_strings):
        for ep in expected:
            if any(ep in rs or rs in ep for rs in rel_str_set):
                rels.add(pid)
                break
    relevant_docs[f'q{i}'] = rels

print('sample: query 0 -> ', queries_map['q0'])
print('  relevant docs:', list(relevant_docs['q0'])[:5])
print('  avg relevant/query:', sum(len(v) for v in relevant_docs.values()) / len(relevant_docs))

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries_map,
    corpus=corpus_map,
    relevant_docs=relevant_docs,
    name='dev_subset',
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 5, 10],
    precision_recall_at_k=[5, 10],
    show_progress_bar=False,
    corpus_chunk_size=5000,
)
print(f'evaluator ready: {len(queries_map)} queries, {len(corpus_map)} corpus')

## 6. Build InputExamples + DataLoader

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

random.seed(42)
examples = [InputExample(texts=[r['text_a'], r['text_b']]) for r in positives]
random.shuffle(examples)

BATCH = 32
loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
print(f'{len(examples)} positives, {len(loader)} steps/epoch @ batch {BATCH}')

## 7. Load BGE-m3, configure loss, measure baseline peak VRAM

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses

BASE = 'BAAI/bge-m3'
print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
print(f'total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = SentenceTransformer(BASE)
model.max_seq_length = 128
train_loss = losses.MultipleNegativesRankingLoss(model)

EPOCHS = 2
CHECKPOINT_STEPS = 200
steps_per_epoch = len(loader)
total_steps = steps_per_epoch * EPOCHS
WARMUP = int(total_steps * 0.1)
print(f'steps/epoch: {steps_per_epoch}, total steps: {total_steps}, warmup: {WARMUP}')
print(f'checkpoint every {CHECKPOINT_STEPS} steps -> ~{total_steps // CHECKPOINT_STEPS} checkpoints')

## 8. Train with periodic evaluation + checkpointing

`model.fit(...)` automatically saves the best checkpoint (by evaluator score) to `output_path` when `evaluator` is supplied. Intermediate checkpoints land in `checkpoint_path/`.

In [ ]:
OUT_DIR = 'auto-parts-search-v3'       # best model by dev MRR@10 lands here
CHKPT_DIR = 'v3-checkpoints'            # all intermediate checkpoints here

model.fit(
    train_objectives=[(loader, train_loss)],
    evaluator=ir_evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={'lr': 2e-5},
    output_path=OUT_DIR,
    save_best_model=True,
    evaluation_steps=CHECKPOINT_STEPS,
    checkpoint_path=CHKPT_DIR,
    checkpoint_save_steps=CHECKPOINT_STEPS,
    checkpoint_save_total_limit=6,
    show_progress_bar=True,
    use_amp=True,
)
print('training done.')
print(f'best checkpoint: {OUT_DIR}')
print(f'peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

## 9. Re-confirm best model beats v1.2 on dev subset before pushing

In [ ]:
best_model = SentenceTransformer(OUT_DIR)
scores = ir_evaluator(best_model)
print('dev subset scores:')
for k, v in scores.items():
    print(f'  {k}: {v:.4f}')

## 10. Push best checkpoint to HF

In [ ]:
from huggingface_hub import HfApi

MODEL_REPO = 'ManmohanBuildsProducts/auto-parts-search-v3'
api = HfApi()
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=MODEL_REPO,
    commit_message=f'v3 — BGE-m3 + MNR, 2ep + best-on-dev, batch 32, {len(examples)} positives',
)
print(f'pushed to https://huggingface.co/{MODEL_REPO}')

## Next (locally)

Full dev + joint-pool benchmark against v1.2:

```bash
# full dev (substring-match relevance)
python3.11 -m training.evaluate \
    --model ManmohanBuildsProducts/auto-parts-search-v3 \
    --benchmark data/training/golden/benchmark_dev.json \
    --out data/training/experiments/2026-04-13-kg-pairs/v3-dev.json

# Fair judged comparison — union v1.2 + v3 top-20, LLM-judge the union, rescore both:
export DEEPSEEK_API_KEY=...
python3.11 -m scripts.judge_benchmark \
    --benchmark data/training/golden/benchmark_dev.json \
    --model ManmohanBuildsProducts/auto-parts-search-v1 \
    --model ManmohanBuildsProducts/auto-parts-search-v3 \
    --deepseek-model deepseek-chat \
    --out data/training/experiments/.../benchmark_dev_graded_joint_v3.jsonl

python3.11 -m training.evaluate_graded --model ManmohanBuildsProducts/auto-parts-search-v1 \
    --graded .../benchmark_dev_graded_joint_v3.jsonl --out .../v1.2-joint-v3.json
python3.11 -m training.evaluate_graded --model ManmohanBuildsProducts/auto-parts-search-v3 \
    --graded .../benchmark_dev_graded_joint_v3.jsonl --out .../v3-joint.json
```

Decision gate: v3 graded nDCG@10 ≥ **0.600** (v1.2 is 0.544, bar +10%) to promote v3 as production. If v3 fails, v1.2 stays and we ship — move to T305 (external bench vs OpenAI/Cohere).